# Transfer Learning — Comparative Analysis
**Models:** ResNet50 · ResNet101 · MobileNetV3Large  
**Dataset:** CIFAR-10 (32×32 RGB, 10 classes)


## 1. Imports

In [ ]:
import os, time, json
import numpy as np
import tensorflow as tf
from matplotlib import pyplot as plt

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Parameters

In [ ]:
BATCH_SIZE = 64
EPOCHS     = 4
classes    = ['airplane','automobile','bird','cat','deer',
               'dog','frog','horse','ship','truck']

# ── Visualization helpers ────────────────────────────────────────────────────
plt.rc('image', cmap='gray')
plt.rc('axes',  facecolor='F8F8F8', titlesize='large', edgecolor='white')
plt.rc('figure',facecolor='F0F0F0')

def display_images(digits, predictions, labels, title):
    n = 10
    idxs = np.random.choice(len(predictions), size=n)
    fig  = plt.figure(figsize=(20, 4))
    plt.title(title); plt.yticks([]); plt.xticks([])
    for i, idx in enumerate(idxs):
        ax = fig.add_subplot(1, 10, i+1)
        pred_cls  = int(predictions[idx])
        true_cls  = int(labels[idx])
        color     = 'green' if pred_cls == true_cls else 'red'
        plt.xlabel(classes[pred_cls], color=color, fontsize=8)
        plt.xticks([]); plt.yticks([])
        plt.imshow(digits[idx])
    plt.tight_layout()
    plt.show()

def plot_metrics(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle(f'{model_name} — Training Curves', fontsize=14, fontweight='bold')
    for ax, metric, title, ylim in zip(
            axes,
            ['loss',     'accuracy'],
            ['Loss',     'Accuracy'],
            [None,       (0, 1)]):
        ax.plot(history.history[metric],        color='royalblue', label=f'Train {title}')
        ax.plot(history.history['val_'+metric], color='tomato',    label=f'Val   {title}')
        ax.set_title(title); ax.legend(); ax.set_xlabel('Epoch')
        if ylim: ax.set_ylim(ylim)
    plt.tight_layout()
    plt.show()

## 3. Load & Pre-process CIFAR-10

In [ ]:
(train_images, train_labels), (val_images, val_labels) = \
    tf.keras.datasets.cifar10.load_data()

print('Train:', train_images.shape, '| Val:', val_images.shape)

In [ ]:
# ── Pre-process for ResNet-family (same for ResNet50 & ResNet101) ────────────
def preprocess_resnet(imgs):
    imgs = imgs.astype('float32')
    return tf.keras.applications.resnet50.preprocess_input(imgs)

# ── Pre-process for MobileNetV3 (expects [0,1] inputs) ──────────────────────
def preprocess_mobilenet(imgs):
    imgs = imgs.astype('float32') / 255.0
    return imgs

# ResNet-family splits
rn_train_X = preprocess_resnet(train_images)
rn_val_X   = preprocess_resnet(val_images)

# MobileNet splits
mn_train_X = preprocess_mobilenet(train_images)
mn_val_X   = preprocess_mobilenet(val_images)

print('Pre-processing done.')

---
## 4. Model Builders

All three models share the same classifier head and training config so results are directly comparable.

| Backbone | Input size | Upsample factor | Weights |
|---|---|---|---|
| ResNet50 | 224×224 | 7× | imagenet |
| ResNet101 | 224×224 | 7× | imagenet |
| MobileNetV3Large | 224×224 | 7× | imagenet |

In [ ]:
def classifier_head(x):
    """Shared dense classification head (10 classes)."""
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Flatten()(x)
    x = tf.keras.layers.Dense(1024, activation='relu')(x)
    x = tf.keras.layers.Dense(512,  activation='relu')(x)
    x = tf.keras.layers.Dense(10,   activation='softmax', name='classification')(x)
    return x


def build_model(backbone_fn, input_shape=(32, 32, 3), upsample=(7, 7)):
    """Generic builder: upsample → backbone → classifier head."""
    inputs  = tf.keras.layers.Input(shape=input_shape)
    x       = tf.keras.layers.UpSampling2D(size=upsample)(inputs)
    x       = backbone_fn(x)
    outputs = classifier_head(x)
    model   = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(
        optimizer='SGD',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy'])
    return model


# ── Backbone factories ───────────────────────────────────────────────────────
def resnet50_backbone(inputs):
    base = tf.keras.applications.ResNet50(
        input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    return base(inputs)

def resnet101_backbone(inputs):
    base = tf.keras.applications.ResNet101(
        input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    return base(inputs)

def mobilenetv3_backbone(inputs):
    base = tf.keras.applications.MobileNetV3Large(
        input_shape=(224, 224, 3), include_top=False, weights='imagenet')
    return base(inputs)

---
## 5. ResNet50 — Build, Train, Evaluate

In [ ]:
print('=' * 60)
print('Building ResNet50 model …')
resnet50_model = build_model(resnet50_backbone)
resnet50_model.summary()

In [ ]:
t0 = time.time()
history_resnet50 = resnet50_model.fit(
    rn_train_X, train_labels,
    epochs=EPOCHS,
    validation_data=(rn_val_X, val_labels),
    batch_size=BATCH_SIZE)
resnet50_time = time.time() - t0
print(f'ResNet50 training time: {resnet50_time/60:.1f} min')

In [ ]:
resnet50_loss, resnet50_acc = resnet50_model.evaluate(
    rn_val_X, val_labels, batch_size=BATCH_SIZE)
print(f'ResNet50  →  Loss: {resnet50_loss:.4f}  |  Accuracy: {resnet50_acc:.4f}')
plot_metrics(history_resnet50, 'ResNet50')

In [ ]:
probs_rn50 = resnet50_model.predict(rn_val_X, batch_size=BATCH_SIZE)
preds_rn50 = np.argmax(probs_rn50, axis=1)
display_images(val_images, preds_rn50, val_labels, 'ResNet50 Predictions (green=correct, red=wrong)')

---
## 6. ResNet101 — Build, Train, Evaluate

In [ ]:
print('=' * 60)
print('Building ResNet101 model …')
resnet101_model = build_model(resnet101_backbone)
resnet101_model.summary()

In [ ]:
t0 = time.time()
history_resnet101 = resnet101_model.fit(
    rn_train_X, train_labels,
    epochs=EPOCHS,
    validation_data=(rn_val_X, val_labels),
    batch_size=BATCH_SIZE)
resnet101_time = time.time() - t0
print(f'ResNet101 training time: {resnet101_time/60:.1f} min')

In [ ]:
resnet101_loss, resnet101_acc = resnet101_model.evaluate(
    rn_val_X, val_labels, batch_size=BATCH_SIZE)
print(f'ResNet101 →  Loss: {resnet101_loss:.4f}  |  Accuracy: {resnet101_acc:.4f}')
plot_metrics(history_resnet101, 'ResNet101')

In [ ]:
probs_rn101 = resnet101_model.predict(rn_val_X, batch_size=BATCH_SIZE)
preds_rn101 = np.argmax(probs_rn101, axis=1)
display_images(val_images, preds_rn101, val_labels, 'ResNet101 Predictions (green=correct, red=wrong)')

---
## 7. MobileNetV3Large — Build, Train, Evaluate

In [ ]:
print('=' * 60)
print('Building MobileNetV3Large model …')
mobilenet_model = build_model(mobilenetv3_backbone)
mobilenet_model.summary()

In [ ]:
t0 = time.time()
history_mobilenet = mobilenet_model.fit(
    mn_train_X, train_labels,
    epochs=EPOCHS,
    validation_data=(mn_val_X, val_labels),
    batch_size=BATCH_SIZE)
mobilenet_time = time.time() - t0
print(f'MobileNetV3 training time: {mobilenet_time/60:.1f} min')

In [ ]:
mobilenet_loss, mobilenet_acc = mobilenet_model.evaluate(
    mn_val_X, val_labels, batch_size=BATCH_SIZE)
print(f'MobileNetV3 →  Loss: {mobilenet_loss:.4f}  |  Accuracy: {mobilenet_acc:.4f}')
plot_metrics(history_mobilenet, 'MobileNetV3Large')

In [ ]:
probs_mn = mobilenet_model.predict(mn_val_X, batch_size=BATCH_SIZE)
preds_mn = np.argmax(probs_mn, axis=1)
display_images(val_images, preds_mn, val_labels, 'MobileNetV3 Predictions (green=correct, red=wrong)')

---
## 8. Side-by-Side Comparative Summary

In [ ]:
# ── 8-A  Accuracy & Loss comparison ─────────────────────────────────────────
models     = ['ResNet50', 'ResNet101', 'MobileNetV3']
val_accs   = [resnet50_acc,   resnet101_acc,   mobilenet_acc]
val_losses = [resnet50_loss,  resnet101_loss,  mobilenet_loss]
colors     = ['#4C9BE8', '#E87A4C', '#4CE87A']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Comparison — CIFAR-10 (4 epochs)', fontsize=15, fontweight='bold')

axes[0].bar(models, val_accs,   color=colors, edgecolor='white', linewidth=1.5)
axes[0].set_title('Validation Accuracy'); axes[0].set_ylim(0, 1)
for i, v in enumerate(val_accs):
    axes[0].text(i, v+0.01, f'{v:.3f}', ha='center', fontweight='bold')

axes[1].bar(models, val_losses, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Validation Loss')
for i, v in enumerate(val_losses):
    axes[1].text(i, v+0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ── 8-B  Overfitting analysis (train vs val accuracy across epochs) ──────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Overfitting Monitor — Train vs Validation Accuracy', fontsize=14, fontweight='bold')

pairs = [
    (history_resnet50,  'ResNet50',      '#4C9BE8'),
    (history_resnet101, 'ResNet101',     '#E87A4C'),
    (history_mobilenet, 'MobileNetV3',   '#4CE87A'),
]

for ax, (hist, name, col) in zip(axes, pairs):
    tr_acc  = hist.history['accuracy']
    val_acc = hist.history['val_accuracy']
    gap     = [t - v for t, v in zip(tr_acc, val_acc)]
    ax.plot(tr_acc,  color=col,  linestyle='-',  linewidth=2.5, label='Train')
    ax.plot(val_acc, color=col,  linestyle='--', linewidth=2.5, label='Val')
    ax.fill_between(range(len(gap)), val_acc, tr_acc,
                    alpha=0.15, color=col, label=f'Gap (avg={np.mean(gap):.3f})')
    ax.set_title(name); ax.set_ylim(0, 1)
    ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── Overfitting verdict ──────────────────────────────────────────────────────
print('\n── Overfitting Verdict ────────────────────────────────────────')
for hist, name in zip(
        [history_resnet50, history_resnet101, history_mobilenet],
        ['ResNet50', 'ResNet101', 'MobileNetV3']):
    tr  = hist.history['accuracy'][-1]
    val = hist.history['val_accuracy'][-1]
    gap = tr - val
    if gap > 0.10:
        verdict = '⚠️  Overfitting  (gap > 10%)'
    elif gap > 0.05:
        verdict = '🟡 Mild overfit  (gap 5–10%)'
    else:
        verdict = '✅ Well-generalised (gap ≤ 5%)'
    print(f'{name:<16}  Train={tr:.3f}  Val={val:.3f}  Gap={gap:.3f}  → {verdict}')

In [ ]:
# ── 8-C  Printable summary table ────────────────────────────────────────────
print('\n' + '='*72)
print(f"{'Model':<18} {'Params':>10} {'Val Acc':>9} {'Val Loss':>10} {'Train Time':>12}")
print('-'*72)
rows = [
    ('ResNet50',    resnet50_model.count_params(),  resnet50_acc,  resnet50_loss,  resnet50_time),
    ('ResNet101',   resnet101_model.count_params(), resnet101_acc, resnet101_loss, resnet101_time),
    ('MobileNetV3', mobilenet_model.count_params(), mobilenet_acc, mobilenet_loss, mobilenet_time),
]
for name, params, acc, loss, t in rows:
    print(f"{name:<18} {params:>10,} {acc:>9.4f} {loss:>10.4f} {t/60:>10.1f} min")
print('='*72)